<a href="https://colab.research.google.com/github/santed7/Data-Science-Cohort-20/blob/main/Project_3_Chinook_SQLite_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chinook SQLite Database — Exploration Notebook

**Course:** CNM Data Science  
**Author:** Dr. Ted (`santed7`)  
**Environment:** Google Colab

This notebook sets up the Chinook SQLite database in Colab, connects to it from Python, and answers the starter exploration questions:

1. What sort of information is in this dataset?
2. How many records are there (per table and overall)?
3. How many different countries / states / cities have records in this dataset?

The notebook is organized so additional questions can be appended in their own sections without disturbing the setup.


## AI Assistance Statement


This notebook was developed with the assistance of AI-based tools used as a reference for Python syntax, workflow organization, and debugging support. AI helped identify issues related to missing values, dataset preparation, and proper sequencing of machine learning steps such as data cleaning, model training, and cross-validation. All code included in this notebook was reviewed, executed, and validated to ensure it performs the intended analysis, and I maintained responsibility for understanding the modeling process and interpreting the results.

## 1. Install SQLite and download the Chinook database

Colab's Ubuntu image already ships with `sqlite3`, but we run the `apt-get` line anyway so the notebook is fully reproducible — if a future Colab image drops it, the install step will silently put it back. Python's standard library `sqlite3` module is what we'll actually use to query the DB; the command-line tool is just a convenience.


In [ ]:
# Install the sqlite3 command-line client (the Python sqlite3 module is already bundled with Python).
# -y answers 'yes' to any prompts; -qq makes apt very quiet so the cell output stays readable.
!apt-get install -y -qq sqlite3

# Confirm the CLI is available; if this prints a version string, we're good.
!sqlite3 --version


### Download the Chinook SQLite database

We pull the database directly from the official `lerocha/chinook-database` GitHub repo. The file `Chinook_Sqlite.sqlite` is a single self-contained SQLite database — once downloaded, no further setup is needed.

We save it to `/content/` (Colab's working directory) so it's easy to reference. If you'd rather keep it in Drive for persistence across sessions, mount Drive (cell below) and copy it to `/content/drive/MyDrive/`.


In [ ]:
# Download the Chinook SQLite database from the official repo.
# -L follows redirects (GitHub raw URLs sometimes redirect).
# -o specifies the output filename so we know exactly where it lands.
!curl -L -o /content/Chinook_Sqlite.sqlite \
    https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite

# Confirm the file landed and check its size (~1 MB is expected).
!ls -lh /content/Chinook_Sqlite.sqlite


**Optional:** mount Google Drive if you want to save outputs (CSV exports, plots) to your Drive folder. Skip this cell if you're just exploring.


In [ ]:
# Mount Google Drive so we can write outputs to /content/drive/MyDrive/.
# This will prompt for permission the first time it runs in a session.
from google.colab import drive
drive.mount('/content/drive')


## 2. Connect to the database from Python

We use Python's built-in `sqlite3` module to open a connection and pandas to pull query results into DataFrames. Pandas DataFrames are nicer to inspect in a notebook than raw cursor rows, and they make follow-up analysis (grouping, plotting, exporting) trivial.


In [ ]:
# Standard imports for SQLite work in a notebook environment.
import sqlite3        # Python's built-in interface to SQLite databases.
import pandas as pd   # For reading query results into DataFrames.

# Path to the Chinook database file we downloaded above.
# Keeping this as a variable means we only update it in one place if the path changes.
DB_PATH = '/content/Chinook_Sqlite.sqlite'

# Open a connection. SQLite connections are lightweight; we keep one open
# for the whole notebook and close it at the end.
conn = sqlite3.connect(DB_PATH)

# Tiny helper: run any SQL string and return a DataFrame.
# We define it once here so every query cell below stays short and readable.
def q(sql):
    # pd.read_sql_query handles the cursor mechanics and column-naming for us.
    return pd.read_sql_query(sql, conn)

print('Connected to:', DB_PATH)


## 3. Question 1 — What sort of information is in this dataset?

**Plan:** list every table in the database, then look at each table's columns. Together, those two pieces of information tell us what entities the dataset tracks (customers, invoices, tracks, etc.) and what attributes are stored about each one.

SQLite stores its catalog in a system table called `sqlite_master`. Filtering it for `type='table'` gives us the user tables.


In [ ]:
# List all user tables in the Chinook database.
# sqlite_master is SQLite's internal catalog; filtering type='table' excludes
# indexes, views, and triggers so we only see actual data tables.
tables_df = q("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
""")

# Expected: 11 tables (Album, Artist, Customer, Employee, Genre, Invoice,
# InvoiceLine, MediaType, Playlist, PlaylistTrack, Track).
print('Number of tables:', len(tables_df))
tables_df


In [ ]:
# For each table, pull its column names and types using PRAGMA table_info.
# PRAGMA is SQLite's mechanism for asking the engine about itself; table_info
# returns one row per column with name, declared type, and PK flag.
schema_rows = []  # We'll collect (table, column, type) tuples and put them in one DataFrame at the end.

for table_name in tables_df['name']:
    # PRAGMA does not accept parameter binding the same way SELECT does, so we
    # interpolate the table name. This is safe here because the names came
    # straight from sqlite_master, not from user input.
    info = q('PRAGMA table_info(' + table_name + ')')
    for _, row in info.iterrows():
        schema_rows.append({
            'table':  table_name,
            'column': row['name'],
            'type':   row['type'],
            'is_pk':  bool(row['pk']),  # 1/0 -> True/False for readability.
        })

schema_df = pd.DataFrame(schema_rows)

# Show the full schema. This is the answer to 'what info is in this dataset':
# the table list tells us *what* is tracked, the column list tells us *which attributes*.
print('Total columns across all tables:', len(schema_df))
schema_df


**Interpretation:** Chinook is a sample digital media store database. It tracks **artists** and their **albums**, the **tracks** on each album (with **genre** and **media type** metadata), the **playlists** those tracks belong to, the **customers** who buy music, the **employees** who support those customers, and the **invoices** (with **invoice lines**) that record each sale. In short — it's a miniature iTunes-style business model, ideal for practicing joins and aggregations.


## 4. Question 2 — How many records are there?

**Plan:** loop through every table and run `SELECT COUNT(*)`, then put the results in a single sorted DataFrame so we can see at a glance which tables are big (probably Track, InvoiceLine) and which are small (probably MediaType, Genre).


In [ ]:
# Count rows in each table. We build a list of dicts then convert to a DataFrame
# at the end -- this is faster than appending to a DataFrame inside the loop.
row_counts = []

for table_name in tables_df['name']:
    # COUNT(*) is the canonical way to get row totals; it's optimized in SQLite.
    count_df = q('SELECT COUNT(*) AS n FROM ' + table_name)
    row_counts.append({
        'table': table_name,
        'records': int(count_df['n'].iloc[0]),  # Pull the single scalar out.
    })

# Sort by row count descending so the largest tables appear first.
counts_df = pd.DataFrame(row_counts).sort_values('records', ascending=False).reset_index(drop=True)

# Total records across the whole database -- handy summary number.
total_records = counts_df['records'].sum()
print('Total records across all tables:', total_records)

counts_df


## 5. Question 3 — How many different countries, states, and cities are represented?

**Plan:** geography in Chinook lives on the **Customer** and **Employee** tables (both have `Country`, `State`, and `City` columns). We'll count distinct values in each, separately for customers and employees, then combine them for an overall picture.

We use `COUNT(DISTINCT ...)` rather than fetching all values and de-duplicating in Python — it's the right tool for the job and pushes the work down to the SQL engine.


In [ ]:
# Distinct country / state / city counts in the Customer table.
# COUNT(DISTINCT col) automatically ignores NULLs, which is what we want --
# a customer with no recorded state shouldn't inflate the state count.
customer_geo = q("""
    SELECT
        COUNT(DISTINCT Country) AS n_countries,
        COUNT(DISTINCT State)   AS n_states,
        COUNT(DISTINCT City)    AS n_cities
    FROM Customer
""")

print('Customer geography:')
customer_geo


In [ ]:
# Same query against the Employee table.
# Chinook's Employee table is small (8 rows, all in Canada in the stock data),
# so the numbers here will be modest -- but it's worth checking.
employee_geo = q("""
    SELECT
        COUNT(DISTINCT Country) AS n_countries,
        COUNT(DISTINCT State)   AS n_states,
        COUNT(DISTINCT City)    AS n_cities
    FROM Employee
""")

print('Employee geography:')
employee_geo


In [ ]:
# Combined view: union the geography rows from both tables, then count distincts.
# UNION (without ALL) automatically removes duplicates, which is exactly what we
# want when asking 'how many different countries are represented in the dataset overall'.
combined_geo = q("""
    WITH all_locations AS (
        SELECT Country, State, City FROM Customer
        UNION
        SELECT Country, State, City FROM Employee
    )
    SELECT
        COUNT(DISTINCT Country) AS n_countries,
        COUNT(DISTINCT State)   AS n_states,
        COUNT(DISTINCT City)    AS n_cities
    FROM all_locations
""")

print('Combined (Customer + Employee) geography:')
combined_geo


In [ ]:
# Bonus: list the actual countries represented, with how many customers come from each.
# This is the kind of breakdown an instructor often wants to see along with the raw counts.
countries_breakdown = q("""
    SELECT
        Country,
        COUNT(*) AS n_customers
    FROM Customer
    GROUP BY Country
    ORDER BY n_customers DESC, Country
""")

print('Number of customer countries:', len(countries_breakdown))
countries_breakdown


## 6. Additional questions

Add new question sections below as you work through the assignment. Each question should follow the same pattern:

1. A markdown cell stating the question and the plan.
2. One or more code cells running the SQL with `q("...")` and showing the result.
3. A short markdown interpretation underneath.

Example template:


### Question 4 — _your question here_

**Plan:** _describe the approach in one or two sentences._


In [ ]:
# Replace this with your SQL.
# Example: top 10 best-selling tracks by number of times they appear on invoices.
example_df = q("""
    SELECT
        t.Name AS track,
        ar.Name AS artist,
        COUNT(*) AS times_sold
    FROM InvoiceLine il
    JOIN Track t  ON il.TrackId = t.TrackId
    JOIN Album al ON t.AlbumId  = al.AlbumId
    JOIN Artist ar ON al.ArtistId = ar.ArtistId
    GROUP BY t.TrackId
    ORDER BY times_sold DESC
    LIMIT 10
""")

example_df


## 7. Close the connection

Always close the SQLite connection when you're done — it releases the file lock and is just good hygiene. If you're still iterating, leave this cell for last.


In [ ]:
# Close the database connection cleanly.
conn.close()
print('Connection closed.')
